In [1]:
import pandas as pd

panel = pd.read_csv(r"C:\Users\nmqui\Downloads\municipal_panel_2000_2025.csv")
income = pd.read_csv(r"C:\Users\nmqui\Downloads\DOR_Income_EQV_Per_Capita2024.csv", thousands=",")

income = income.rename(columns={"DOR Code": "dor_code",
                                "DOR Income Per Capita": "income_per_capita",
                                "EQV Per Capita": "eqv_per_capita"})
income = income[["dor_code", "income_per_capita", "eqv_per_capita"]]

df = panel[panel["year"] == 2024].merge(income, on="dor_code", how="left")

gateway = ["Attleboro","Barnstable","Brockton","Chelsea","Chicopee","Everett","Fall River","Fitchburg",
           "Haverhill","Holyoke","Lawrence","Leominster","Lowell","Lynn","Malden","Methuen","New Bedford",
           "Peabody","Pittsfield","Quincy","Revere","Salem","Springfield","Taunton","Westfield","Worcester"]
df["gateway"] = df["municipality"].isin(gateway)

print("Rows:", len(df), "| Missing income:", df["income_per_capita"].isna().sum())
print("Gateway Cities found:", df["gateway"].sum(), "of 26")

Rows: 351 | Missing income: 0
Gateway Cities found: 26 of 26


In [2]:
peers = df[(df["population"] >= 35000) & (df["population"] <= 250000)].copy()

# Drop towns with no FY2024 finance report (their spending shows as $0)
missing = peers.loc[peers["finance_zero_flag"] == 1, "municipality"].tolist()
peers = peers[peers["finance_zero_flag"] == 0]
print("Dropped (no FY2024 finance data):", missing)

peers["spending_per_resident"] = peers["expenditure_total_expenditures"] / peers["population"]
peers["gen_gov_share_pct"] = peers["expenditure_general_government"] / peers["expenditure_total_expenditures"] * 100
peers["group"] = peers["gateway"].map({True: "Gateway City", False: "Other city (35k-250k)"})

print(peers["group"].value_counts().to_string(), "\n")
summary = peers.groupby("group")[["income_per_capita", "spending_per_resident", "gen_gov_share_pct"]].median().round(1)
print(summary.to_string())

Dropped (no FY2024 finance data): ['Holyoke']
group
Gateway City             25
Other city (35k-250k)    22 

                       income_per_capita  spending_per_resident  gen_gov_share_pct
group                                                                             
Gateway City                     28282.0                 4136.9                3.8
Other city (35k-250k)            50942.5                 4340.2                4.8


In [3]:
cols = ["dor_code", "municipality", "county", "population", "group", "income_per_capita",
        "eqv_per_capita", "expenditure_total_expenditures", "spending_per_resident", "gen_gov_share_pct"]
peers[cols].to_csv(r"C:\Users\nmqui\Downloads\gateway_vs_peers_FY2024.csv", index=False)
print(len(peers), "cities saved")

47 cities saved
